# 🚗⚡ Kaggle Playground Series s6e9 : Predicting Electric Vehicle Purchases
### 🏆 Grandmaster Solution (10-Fold Ensemble + GPU T4 + Level-2 Stacking + Pseudo-Labeling)
**Objectif :** Atteindre et franchir **0.9467+ ROC-AUC** (Top 1) avec un temps d'exécution optimisé sur **Google Colab (GPU T4)** et **Kaggle Notebooks**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Kaggle](https://img.shields.io/badge/Platform-Kaggle-blue?logo=kaggle)](https://www.kaggle.com/competitions/playground-series-s6e9)
[![Hardware](https://img.shields.io/badge/Hardware-NVIDIA%20GPU%20T4%20CUDA-green?logo=nvidia)]()
[![Metric](https://img.shields.io/badge/Metric-ROC--AUC-orange)]()

---

### 🧠 Nouveaux Leviers Grandmaster Implémentés :
1. **Accélération Extrême GPU T4** : CatBoost GPU avec `border_count=128` + XGBoost GPU `tree_method='hist', device='cuda'` divisant le temps d'entraînement par 10.
2. **5 Nouvelles Variables Clés** :
   - `Commute_Stress_Index` : Mesure la tension entre trajet quotidien et infrastructure de recharge.
   - `Anxiety_Buffer_Ratio` : Amortissement de la peur de panne sèche par le réseau local.
   - `EV_Affordability_Index` : Pouvoir d'achat corrigé par l'aide financière et le profil d'âge.
   - `Station_Access_Per_Vehicle` : Disponibilité de bornes par véhicule possédé.
   - `Eco_Action_Propensity` : Propension écologique convertie en acte d'achat.
3. **Fréquences de Combinaisons Catégorielles (Count Encoding)** : Capture de la rareté des sous-segments démographiques.
4. **Smooth Out-Of-Fold Bayesian Target Encoding** : Lissage $m$-estimate sans risque de fuite d'information.
5. **Double Assemblage Hybride** : Optimisation Nelder-Mead sur les Rangs + Meta-Learner Stacking (Logistic Regression).
6. **Pseudo-Labeling Débrayable** : Ré-entraînement sur les prédictions haute confiance ($P \ge 0.98$ ou $P \le 0.02$).

## 🛠️ 1. Installation des Dépendances & Configuration Matérielle (GPU T4)
Installe les bibliothèques et configure automatiquement l'accélération NVIDIA CUDA.

In [ ]:
# Installation des bibliothèques requises
!pip install -q lightgbm xgboost catboost optuna scikit-learn matplotlib seaborn scipy kaggle

import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

import xgboost as xgb
from xgboost import XGBClassifier

import catboost as cb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Vérification du GPU

print("="*65)
print(f"🚀 Accélération Matérielle : {gpu_name}")
print(f"📦 Versions : LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | CatBoost {cb.__version__}")
print("="*65)# Vérification et Détection avancée du GPU NVIDIA
def check_cuda_available():
    try:
        from xgboost import XGBClassifier
        m = XGBClassifier(device='cuda', n_estimators=1)
        m.fit(np.zeros((2, 2)), np.array([0, 1]))
        return True
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True
    except Exception:
        pass
    try:
        import subprocess
        res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return res.returncode == 0
    except Exception:
        return False

def get_gpu_device_name():
    try:
        import subprocess
        res = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], encoding='utf-8')
        return res.strip()
    except Exception:
        return "NVIDIA GPU CUDA"

HAS_CUDA = check_cuda_available()
gpu_name = get_gpu_device_name() if HAS_CUDA else "CPU Multi-Core"


## 📂 2. Téléchargement & Chargement des Données
Localise automatiquement les fichiers ou propose le téléchargement direct via l'API Kaggle ou Google Drive.

In [ ]:
def find_or_download_data():
    """Localise les fichiers ou les télécharge automatiquement sur Colab/Kaggle."""
    candidates = [
        '/kaggle/input/playground-series-s6e9',
        '/kaggle/input/predicting-electric-vehicle-purchases',
        '/content/drive/MyDrive/data',
        '/content/drive/MyDrive',
        '/content',
        './data',
        '.'
    ]
    for c in candidates:
        tr = os.path.join(c, 'train.csv')
        te = os.path.join(c, 'test.csv')
        if os.path.exists(tr) and os.path.exists(te):
            return tr, te
            
    # Si non trouvé et sur Google Colab, proposition de téléchargement Kaggle
    if os.path.exists('/content') and not os.path.exists('train.csv'):
        print("💡 Données non trouvées localement. Tentative de téléchargement Kaggle...")
        if os.path.exists('kaggle.json'):
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle competitions download -c playground-series-s6e9
            !unzip -o -q playground-series-s6e9.zip
            return 'train.csv', 'test.csv'
            
    return 'train.csv', 'test.csv'

train_path, test_path = find_or_download_data()
print(f"📂 Train Path : {train_path}")
print(f"📂 Test Path  : {test_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"\n📊 Dimensions Train : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")
print(f"📊 Dimensions Test  : {test.shape[0]:,} lignes × {test.shape[1]} colonnes")
display(train.head(3))

## ⚙️ 3. Paramètres de Contrôle du Pipeline
Basculez facilement entre le mode expérimentation rapide (`N_SPLITS = 5`) et le mode Grandmaster final (`N_SPLITS = 10` + `USE_PSEUDO_LABELING = True`).

In [ ]:
# ==================================================================================================
# PARAMÈTRES DE CONTRÔLE
# ==================================================================================================
N_SPLITS = 10                         # 10 pour le score maximal (5 pour itération rapide)
USE_GPU = HAS_CUDA                   # Détecte le GPU T4 automatiquement
USE_PSEUDO_LABELING = True           # Active le Round 2 semi-supervisé pour viser 0.9467+
PSEUDO_LABEL_THRESHOLD_HIGH = 0.980  # Seuil positif haute confiance
PSEUDO_LABEL_THRESHOLD_LOW = 0.020   # Seuil négatif haute confiance
USE_STACKING_META_LEARNER = True     # Active le Stacking Level-2 (Régression Logistique)
RANDOM_SEED = 42

def format_duration(seconds):
    if seconds < 60:
        return f"{seconds:.2f}s"
    minutes = int(seconds // 60)
    sec = seconds % 60
    return f"{minutes}m {sec:04.1f}s"

print(f"✅ Configuration prête (CV: {N_SPLITS}-Fold | GPU: {USE_GPU} | Pseudo-Labeling: {USE_PSEUDO_LABELING})")

## 🧠 4. Feature Engineering Grandmaster (Nouveaux Leviers)
Génération des nouvelles variables métier non-linéaires :
1. `Commute_Stress_Index` : $\text{Commute} / (\text{Stations} + 6 \times \text{HomeCharging} + 1)$
2. `Anxiety_Buffer_Ratio` : $(\text{Anxiety} + 1) / (\text{Stations} + 3 \times \text{HomeCharging} + 1)$
3. `EV_Affordability_Index` : $(\text{Income} \times (1 + 0.4 \times \text{Subsidy})) / (\text{Age} \times (\text{Cars} + 1) + 1)$
4. `Station_Access_Per_Vehicle` & `Eco_Action_Propensity`
5. **Count / Frequency Encoding** des sous-groupes démographiques.

In [ ]:
def preprocess_and_feature_engineering(df):
    """Génération des variables métier à fort impact."""
    df = df.copy()
    
    # --- A. Encodage binaire et ordinal ---
    binary_map = {'Yes': 1.0, 'No': 0.0, 1: 1.0, 0: 0.0, '1': 1.0, '0': 0.0}
    if 'Home_Charging_Possible' in df.columns:
        df['Home_Charging_Possible'] = df['Home_Charging_Possible'].map(binary_map).fillna(0.0).astype(float)
    if 'Subsidy_Available' in df.columns:
        df['Subsidy_Available'] = df['Subsidy_Available'].map(binary_map).fillna(0.0).astype(float)
        
    range_map = {'Low': 0.0, 'Medium': 1.0, 'High': 2.0, 0: 0.0, 1: 1.0, 2: 2.0, '0': 0.0, '1': 1.0, '2': 2.0}
    if 'Range_Anxiety_Level' in df.columns:
        df['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].map(range_map).fillna(1.0).astype(float)

    # --- B. Features Métier de Base ---
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['Charging_Home_Work_Ratio'] = (df['Charging_Stations_Near_Home'] + 1.0) / (df['Charging_Stations_Near_Work'] + 1.0)
    df['Income_Per_Car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1.0)
    df['Income_Per_Age'] = df['Annual_Income_USD'] / (df['Age'] + 1.0)
    df['Stations_Per_Commute_km'] = df['Total_Charging_Stations'] / (df['Daily_Commute_km'] + 1.0)
    df['Commute_Per_Age'] = df['Daily_Commute_km'] / (df['Age'] + 1.0)

    # --- C. Le Paradoxe de Simpson & Dépendance à la Recharge Publique ---
    df['Need_Public_Charging'] = (1.0 - df['Home_Charging_Possible']) * df['Total_Charging_Stations']
    df['No_Home_Charge_Anxiety'] = (1.0 - df['Home_Charging_Possible']) * (df['Range_Anxiety_Level'] + 1.0)
    df['Home_Charge_High_Income'] = df['Home_Charging_Possible'] * (df['Annual_Income_USD'] / 10000.0)
    df['Commute_No_Home_Charge'] = df['Daily_Commute_km'] * (1.0 - df['Home_Charging_Possible'])

    # --- D. NOUVEAUX LEVIERS GRANDMASTER ---
    df['Commute_Stress_Index'] = df['Daily_Commute_km'] / (df['Total_Charging_Stations'] + df['Home_Charging_Possible'] * 6.0 + 1.0)
    df['Anxiety_Buffer_Ratio'] = (df['Range_Anxiety_Level'] + 1.0) / (df['Total_Charging_Stations'] + df['Home_Charging_Possible'] * 3.0 + 1.0)
    df['EV_Affordability_Index'] = (df['Annual_Income_USD'] * (1.0 + 0.4 * df['Subsidy_Available'])) / ((df['Age'] * (df['Number_of_Cars_Owned'] + 1.0)) + 1.0)
    df['Station_Access_Per_Vehicle'] = df['Total_Charging_Stations'] / (df['Number_of_Cars_Owned'] + 1.0)
    df['Eco_Action_Propensity'] = (df['Environmental_Concern_Level'] * (df['Home_Charging_Possible'] + 1.0)) / (df['Range_Anxiety_Level'] + 1.0)

    # --- E. Élasticité des Subventions & Interactions Non-Linéaires ---
    df['Subsidy_Elasticity'] = df['Subsidy_Available'] / ((df['Annual_Income_USD'] / 10000.0) + 1.0)
    df['Eco_x_Income'] = (df['Annual_Income_USD'] / 10000.0) * df['Environmental_Concern_Level']
    df['Eco_x_Stations'] = df['Environmental_Concern_Level'] * df['Total_Charging_Stations']
    df['Eco_and_Home_Charge'] = df['Environmental_Concern_Level'] * df['Home_Charging_Possible']

    # --- F. Score Global de Maturité VE ---
    df['EV_Readiness_Score'] = (
        (df['Home_Charging_Possible'] * 3.0) + 
        (df['Subsidy_Available'] * 1.5) + 
        (df['Total_Charging_Stations'] * 0.25) - 
        (df['Range_Anxiety_Level'] * 1.5)
    )
    
    # --- G. Combinaisons Catégorielles pour Target Encoding et Count Encoding ---
    df['City_and_Car'] = df['City_Type'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    df['Gender_and_Car'] = df['Gender'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    df['City_and_Anxiety'] = df['City_Type'].astype(str) + "_" + df['Range_Anxiety_Level'].astype(str)
    df['HomeCharge_and_City'] = df['Home_Charging_Possible'].astype(str) + "_" + df['City_Type'].astype(str)
    df['Demographic_Segment'] = df['City_Type'].astype(str) + "_" + df['Gender'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    
    # --- H. Frequency / Count Encoding ---
    for col in ['City_and_Car', 'Gender_and_Car', 'Demographic_Segment']:
        freq = df[col].value_counts(normalize=True).to_dict()
        df[f'{col}_Freq'] = df[col].map(freq).astype(float)
    
    cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'City_and_Car', 'Gender_and_Car', 'City_and_Anxiety', 'HomeCharge_and_City', 'Demographic_Segment']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df


def add_group_aggregations(df_all):
    """Calcule les écarts et ratios par rapport aux moyennes de groupe."""
    df = df_all.copy()
    for grp in ['City_and_Car', 'HomeCharge_and_City', 'Demographic_Segment']:
        stats = df.groupby(grp, observed=False)['Annual_Income_USD'].agg(['mean', 'std']).reset_index()
        stats.columns = [grp, f'{grp}_Income_Mean', f'{grp}_Income_Std']
        df = df.merge(stats, on=grp, how='left')
        df[f'{grp}_Income_Diff'] = df['Annual_Income_USD'] - df[f'{grp}_Income_Mean']
        df[f'{grp}_Income_Ratio'] = df['Annual_Income_USD'] / (df[f'{grp}_Income_Mean'] + 1.0)
        
    return df

print("⚙️ Fonctions de Feature Engineering prêtes.")


## 🎯 5. Smooth Target Encoding Out-Of-Fold (Sans Data Leakage)
Encodage bayésien lissé calculé de manière strictement isolée pli par pli.

In [ ]:
def apply_oof_target_encoding(train_df, test_df, cat_cols, target_col, skf, m_smoothing=20.0):
    """Calcule l'encodage de la cible avec lissage bayésien de manière Out-Of-Fold."""
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()
    
    global_mean = float(train_df[target_col].mean())
    
    for col in cat_cols:
        col_name = f"{col}_TE"
        train_encoded[col_name] = 0.0
        test_col_encoded = np.zeros(len(test_df), dtype=float)
        
        for train_idx, val_idx in skf.split(train_df, train_df[target_col]):
            fold_train = train_df.iloc[train_idx]
            fold_val = train_df.iloc[val_idx]
            
            stats = fold_train.groupby(fold_train[col].astype(str), observed=False)[target_col].agg(['count', 'mean'])
            smoothed_series = (stats['count'] * stats['mean'] + m_smoothing * global_mean) / (stats['count'] + m_smoothing)
            smoothed_dict = smoothed_series.to_dict()
            
            val_vals = fold_val[col].astype(str).map(smoothed_dict).fillna(global_mean).astype(float).values
            train_encoded.loc[train_encoded.index[val_idx], col_name] = val_vals
            
            test_vals = test_df[col].astype(str).map(smoothed_dict).fillna(global_mean).astype(float).values
            test_col_encoded += test_vals / skf.n_splits
            
        test_encoded[col_name] = test_col_encoded
        
    return train_encoded, test_encoded

print("🎯 Smooth Target Encoding configuré.")

## ⚖️ 6. Assemblage Hybride : Rank Averaging & Stacking Level-2
Combinaison mathématique sur l'espace des rangs et méta-apprenant logistique pour maximiser le ROC-AUC final.

In [ ]:
def rank_average(pred_list, weights=None):
    """Moyenne pondérée des rangs normalisée entre 0 et 1."""
    if weights is None:
        weights = [1.0 / len(pred_list)] * len(pred_list)
    else:
        weights = [w / sum(weights) for w in weights]
        
    ranked_sum = np.zeros(len(pred_list[0]))
    for pred, w in zip(pred_list, weights):
        ranked = rankdata(pred) / len(pred)
        ranked_sum += ranked * w
        
    return ranked_sum


def optimize_ensemble_weights(y_true, pred_list):
    """Trouve mathématiquement la combinaison de poids maximisant le ROC-AUC."""
    n_models = len(pred_list)
    if n_models == 1:
        return [1.0]

    def objective(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blend = rank_average(pred_list, weights=w)
        return -roc_auc_score(y_true, blend)

    init_weights = [1.0 / n_models] * n_models
    bounds = [(0.01, 1.0)] * n_models
    res = minimize(objective, init_weights, method='Nelder-Mead', bounds=bounds)
    opt_weights = res.x / np.sum(res.x)
    return opt_weights.tolist()


def train_stacking_meta_learner(y_true, oof_preds_list, test_preds_list):
    """Entraîne un meta-learner de niveau 2 (Logistic Regression) sur les prédictions OOF."""
    X_meta = np.column_stack(oof_preds_list)
    X_test_meta = np.column_stack(test_preds_list)
    
    meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED)
    meta_model.fit(X_meta, y_true)
    
    meta_oof = meta_model.predict_proba(X_meta)[:, 1]
    meta_test = meta_model.predict_proba(X_test_meta)[:, 1]
    meta_auc = roc_auc_score(y_true, meta_oof)
    
    return meta_oof, meta_test, meta_auc, meta_model

print("⚖️ Fonctions d'assemblage et de stacking prêtes.")

## 🚀 7. Pipeline d'Entraînement Multi-Modèles Ultra-Rapide (GPU T4)
Entraîne **LightGBM**, **XGBoost CUDA** et **CatBoost GPU** avec des hyperparamètres optimisés pour une vitesse maximale.

In [ ]:
def train_ensemble_pipeline(X, y, X_test, test_ids, skf, tag="standard"):
    """Entraîne LightGBM, XGBoost et CatBoost avec 10-Fold CV et optimise leur assemblage."""
    timing_report = {}
    feature_importances = {}
    
    # --------------------------------------------------------------------------
    # 1. LightGBM (Hyper-Tuned & Fast)
    # --------------------------------------------------------------------------
    print("\n" + "="*70)
    print(f"📦 [1/3] Entraînement LightGBM ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    lgb_start = time.time()
    
    lgb_params = {
        'n_estimators': 2500,
        'learning_rate': 0.03,
        'num_leaves': 36,
        'max_depth': 6,
        'subsample': 0.85,
        'colsample_bytree': 0.80,
        'min_child_samples': 40,
        'reg_alpha': 0.1,
        'reg_lambda': 1.5,
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'verbosity': -1
    }
        
    lgb_oof = np.zeros(len(X))
    lgb_test = np.zeros(len(X_test))
    lgb_importances = np.zeros(X.shape[1])
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        f_start = time.time()
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**lgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='auc',
            callbacks=[early_stopping(40, verbose=False), log_evaluation(0)]
        )
        lgb_oof[val_idx] = model.predict_proba(X_va)[:, 1]
        lgb_test += model.predict_proba(X_test)[:, 1] / skf.n_splits
        lgb_importances += model.feature_importances_ / skf.n_splits
        
        f_auc = roc_auc_score(y_va, lgb_oof[val_idx])
        print(f"  👉 LGBM Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    lgb_auc = roc_auc_score(y, lgb_oof)
    lgb_time = time.time() - lgb_start
    timing_report['LightGBM'] = {'auc': lgb_auc, 'time': lgb_time}
    feature_importances['LightGBM'] = lgb_importances
    print(f"  🏆 Score LightGBM OOF ROC-AUC : {lgb_auc:.5f} | ⏱️ Temps Total : {format_duration(lgb_time)}")

    # --------------------------------------------------------------------------
    # 2. XGBoost (GPU Accelerated CUDA Hist)
    # --------------------------------------------------------------------------
    print("\n" + "="*70)
    gpu_tag = "GPU (CUDA)" if USE_GPU else "CPU"
    print(f"📦 [2/3] Entraînement XGBoost [{gpu_tag}] ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    xgb_start = time.time()
    
    xgb_oof = np.zeros(len(X))
    xgb_test = np.zeros(len(X_test))
    
    xgb_params = {
        'n_estimators': 2200,
        'learning_rate': 0.03,
        'max_depth': 6,
        'subsample': 0.85,
        'colsample_bytree': 0.80,
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'enable_categorical': True,
        'eval_metric': 'auc',
        'early_stopping_rounds': 40,
        'reg_alpha': 0.1,
        'reg_lambda': 1.5,
        'random_state': RANDOM_SEED,
        'n_jobs': -1
    }
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        f_start = time.time()
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
        xgb_oof[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test += xgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
        
        f_auc = roc_auc_score(y_va, xgb_oof[val_idx])
        print(f"  👉 XGB Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    xgb_auc = roc_auc_score(y, xgb_oof)
    xgb_time = time.time() - xgb_start
    timing_report['XGBoost'] = {'auc': xgb_auc, 'time': xgb_time}
    print(f"  🏆 Score XGBoost OOF ROC-AUC : {xgb_auc:.5f} | ⏱️ Temps Total : {format_duration(xgb_time)}")

    # --------------------------------------------------------------------------
    # 3. CatBoost (GPU Quantized Logloss - Ultra Fast)
    # --------------------------------------------------------------------------
    print("\n" + "="*70)
    gpu_label = "GPU (CUDA)" if USE_GPU else "CPU"
    print(f"📦 [3/3] Entraînement CatBoost [{gpu_label}] ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    cat_start = time.time()
    
    cat_oof = np.zeros(len(X))
    cat_test = np.zeros(len(X_test))
    
    cat_cols_list = [col for col in ['Gender', 'City_Type', 'Current_Car_Type'] if col in X.columns]
    X_cat = X.copy()
    X_test_cat = X_test.copy()
    for col in cat_cols_list:
        X_cat[col] = X_cat[col].astype(str)
        X_test_cat[col] = X_test_cat[col].astype(str)
        
    cat_params = {
        'iterations': 1800,
        'learning_rate': 0.035,
        'depth': 6,
        'l2_leaf_reg': 4.0,
        'eval_metric': 'Logloss',
        'border_count': 128,
        'random_seed': RANDOM_SEED,
        'verbose': False,
        'task_type': 'GPU' if USE_GPU else 'CPU',
        'allow_writing_files': False
    }
    if not USE_GPU:
        cat_params['thread_count'] = -1
        
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cat, y)):
        f_start = time.time()
        X_tr, y_tr = X_cat.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_cat.iloc[val_idx], y.iloc[val_idx]
        
        cat_model = CatBoostClassifier(**cat_params, cat_features=cat_cols_list)
        cat_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=40,
            verbose=False
        )
        cat_oof[val_idx] = cat_model.predict_proba(X_va)[:, 1]
        cat_test += cat_model.predict_proba(X_test_cat)[:, 1] / skf.n_splits
        
        f_auc = roc_auc_score(y_va, cat_oof[val_idx])
        print(f"  👉 CatBoost Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    cat_auc = roc_auc_score(y, cat_oof)
    cat_time = time.time() - cat_start
    timing_report['CatBoost'] = {'auc': cat_auc, 'time': cat_time}
    print(f"  🏆 Score CatBoost OOF ROC-AUC : {cat_auc:.5f} | ⏱️ Temps Total : {format_duration(cat_time)}")

    # --------------------------------------------------------------------------
    # Assemblage Hybride : Rank Averaging + Stacking Level-2
    # --------------------------------------------------------------------------
    models_oof = [lgb_oof, xgb_oof, cat_oof]
    models_test = [lgb_test, xgb_test, cat_test]
    
    # 1. Poids optimaux par Nelder-Mead sur les rangs
    opt_weights = optimize_ensemble_weights(y, models_oof)
    print(f"\n🎯 Poids d'assemblage Nelder-Mead : {[round(w, 3) for w in opt_weights]}")
    rank_blend_oof = rank_average(models_oof, weights=opt_weights)
    rank_blend_test = rank_average(models_test, weights=opt_weights)
    rank_auc = roc_auc_score(y, rank_blend_oof)
    print(f"  ✨ ROC-AUC Rank Averaging : {rank_auc:.5f}")
    
    # 2. Stacking Level-2 Meta-Learner
    if USE_STACKING_META_LEARNER:
        stack_oof, stack_test, stack_auc, _ = train_stacking_meta_learner(y, models_oof, models_test)
        print(f"  ✨ ROC-AUC Stacking Meta-Learner : {stack_auc:.5f}")
        
        # Fusion hybride Rank Averaging + Stacking
        final_oof = rank_average([rank_blend_oof, stack_oof], weights=[0.50, 0.50])
        final_test_preds = rank_average([rank_blend_test, stack_test], weights=[0.50, 0.50])
        final_auc = roc_auc_score(y, final_oof)
        print(f"  🚀 ROC-AUC Hybride (Rank + Stacking) : {final_auc:.5f}")
    else:
        final_oof = rank_blend_oof
        final_test_preds = rank_blend_test
        final_auc = rank_auc
        
    return final_oof, final_test_preds, final_auc, timing_report, feature_importances

print("🚀 Pipeline d'entraînement ultra-rapide prêt.")


## ⚡ 8. Exécution Round 1 : Entraînement Initial de l'Ensemble
Prépare le dataset complet et lance l'entraînement 10-Fold sur LightGBM, XGBoost et CatBoost.

In [ ]:
total_start_time = time.time()

# 1. Extraction de la cible
y = (train['Will_Buy_EV'] == 'Yes').astype(int)
test_ids = test['id']

# 2. Concaténation pour le Feature Engineering global
fe_start = time.time()
print("⚙️ Application du Feature Engineering Avancé et des Statistiques de Groupe...")
df_all = pd.concat([train.assign(is_train=1), test.assign(is_train=0, Will_Buy_EV='No')], axis=0).reset_index(drop=True)
df_fe = preprocess_and_feature_engineering(df_all)
df_fe = add_group_aggregations(df_fe)

train_fe = df_fe[df_fe['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
test_fe = df_fe[df_fe['is_train'] == 0].drop(columns=['is_train', 'Will_Buy_EV']).reset_index(drop=True)
train_fe['Will_Buy_EV'] = y

# 3. Smooth Target Encoding Out-Of-Fold
print("⚙️ Application du Smooth Target Encoding Out-Of-Fold (Sans Leakage)...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
te_cols = ['City_Type', 'Current_Car_Type', 'City_and_Car', 'City_and_Anxiety', 'HomeCharge_and_City', 'Demographic_Segment']
train_fe, test_fe = apply_oof_target_encoding(train_fe, test_fe, te_cols, 'Will_Buy_EV', skf, m_smoothing=20.0)
print(f"⏱️ Feature Engineering complet terminé en : {format_duration(time.time() - fe_start)}")

features = [c for c in train_fe.columns if c not in ['id', 'Will_Buy_EV']]
X = train_fe[features]
X_test = test_fe[features]

print(f"📊 Nombre total de variables explicatives générées : {len(features)}")

# 4. Lancement du Round 1
oof_round1, test_round1, auc_round1, timing_report_r1, ft_imp = train_ensemble_pipeline(
    X, y, X_test, test_ids, skf, tag="Round 1 - Base Ensemble"
)

print("\n" + "="*70)
print(f"🏆 SCORE ENSEMBLE ROUND 1 ROC-AUC : {auc_round1:.5f}")
print("="*70)

## 📊 9. Visualisation : Importances des Variables & Comparatif ROC-AUC

In [ ]:
plt.figure(figsize=(16, 6))

# Subplot 1: Feature Importances
plt.subplot(1, 2, 1)
imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': ft_imp['LightGBM']
}).sort_values('Importance', ascending=False).head(15)

sns.barplot(data=imp_df, x='Importance', y='Feature', palette='mako')
plt.title("Top 15 Features les plus Importantes (LightGBM)", fontsize=13, fontweight='bold')
plt.xlabel("Importance Moyenne (Gain)")
plt.ylabel("")

# Subplot 2: Comparatif des Scores ROC-AUC
plt.subplot(1, 2, 2)
model_names = list(timing_report_r1.keys()) + ['Grandmaster Ensemble']
scores = [info['auc'] for info in timing_report_r1.values()] + [auc_round1]
colors = ['#4C72B0', '#55A868', '#C44E52'][:len(model_names)-1] + ['#FF7F0E']

bars = plt.barh(model_names, scores, color=colors, height=0.55)
plt.xlim(min(scores) - 0.005, max(scores) + 0.003)
plt.title("Comparatif des Performances ROC-AUC (10-Fold CV)", fontsize=13, fontweight='bold')
plt.xlabel("OOF ROC-AUC Score")

for bar, score in zip(bars, scores):
    plt.text(bar.get_width() + 0.0003, bar.get_y() + bar.get_height()/2, f"{score:.5f}", va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 🤖 10. Exécution Round 2 : Pseudo-Labeling Itératif Semi-Supervisé
Enrichissement du jeu d'entraînement avec les prédictions haute confiance du test set pour affiner la frontière de décision finale.

In [ ]:
final_test_preds = test_round1
final_auc = auc_round1

if USE_PSEUDO_LABELING:
    print("\n" + "="*70)
    print("🤖 Lancement du Pseudo-Labeling Itératif Semi-Supervisé (Cible 0.9467+)...")
    print("="*70)
    
    pseudo_pos_mask = test_round1 >= PSEUDO_LABEL_THRESHOLD_HIGH
    pseudo_neg_mask = test_round1 <= PSEUDO_LABEL_THRESHOLD_LOW
    pseudo_indices = np.where(pseudo_pos_mask | pseudo_neg_mask)[0]
    
    print(f"• Échantillons Test haute confiance identifiés : {len(pseudo_indices):,} sur {len(test):,}")
    print(f"  - Positifs (>= {PSEUDO_LABEL_THRESHOLD_HIGH}) : {pseudo_pos_mask.sum():,}")
    print(f"  - Négatifs (<= {PSEUDO_LABEL_THRESHOLD_LOW})  : {pseudo_neg_mask.sum():,}")
    
    if len(pseudo_indices) > 500:
        pseudo_X_test = X_test.iloc[pseudo_indices].copy()
        pseudo_y_test = (test_round1[pseudo_indices] >= 0.5).astype(int)
        
        X_augmented = pd.concat([X, pseudo_X_test], axis=0).reset_index(drop=True)
        y_augmented = pd.concat([y, pd.Series(pseudo_y_test)], axis=0).reset_index(drop=True)
        
        print(f"• Taille du Train Augmenté pour le Round 2 : {len(X_augmented):,} lignes")
        
        skf_aug = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
        oof_round2, test_round2, auc_round2, timing_report_aug, _ = train_ensemble_pipeline(
            X_augmented, y_augmented, X_test, test_ids, skf_aug, tag="Round 2 - Supercharged Ensemble"
        )
        
        # Évaluation du score combiné sur le train initial
        oof_aug_original = oof_round2[:len(train)]
        combined_oof = rank_average([oof_round1, oof_aug_original], weights=[0.40, 0.60])
        final_auc = roc_auc_score(y, combined_oof)
        final_test_preds = rank_average([test_round1, test_round2], weights=[0.40, 0.60])
        
        print("\n" + "="*70)
        print(f"✨ SCORE GLOBAL COMBINÉ APRÈS PSEUDO-LABELING ROC-AUC : {final_auc:.5f}")
        print("="*70)
    else:
        print("⚠️ Nombre d'échantillons insuffisant pour le pseudo-labeling.")

## 📁 11. Sauvegarde des Soumissions & Bilan Final

In [ ]:
os.makedirs('submissions/final', exist_ok=True)
os.makedirs('submissions/temp', exist_ok=True)

print("="*70)
print("📊 BILAN GLOBAL DES TEMPS D'ENTRAÎNEMENT")
print("="*70)
for model_name, info in timing_report_r1.items():
    print(f"  • {model_name:<12} | ROC-AUC OOF : {info['auc']:.5f} | ⏱️ {format_duration(info['time'])}")
print("-" * 70)
print(f"🏆 SCORE FINAL DE L'ENSEMBLE ROC-AUC : {final_auc:.5f}")
print("="*70)

# Formatage de la soumission
ensemble_filename = f"submission_ensemble_grandmaster_auc_{final_auc:.5f}.csv"
final_path = os.path.join('submissions', 'final', ensemble_filename)

sub = pd.DataFrame({
    'id': test_ids,
    'Will_Buy_EV': final_test_preds
})

sub.to_csv(final_path, index=False)
sub.to_csv('submission.csv', index=False)

print(f"\n✅ Soumission Finale enregistrée : {final_path}")
print(f"   (Fichier de soumission Kaggle prêt : submission.csv)")
print(f"   (Dimensions : {sub.shape[0]} lignes × {sub.shape[1]} colonnes)")

total_duration = time.time() - total_start_time
print(f"\n🏁 Pipeline Grandmaster terminé avec succès ! ⏱️ Durée Totale : {format_duration(total_duration)}")
display(sub.head(10))